# Notebook 07 — Baseline Models
## Section 1: Naive + Linear/Ridge Baselines (MAE)

First regression models for **`delay_in_min`**.

**Models**
1. Naive mean predictor  
2. Naive median predictor  
3. Linear Regression (operational features only)  
4. Ridge Regression (operational)  
5. Linear Regression (operational + weather)  
6. Ridge Regression (full features)

**Metric:** MAE only (minutes)  
**Data:** `ice_train.parquet` → train | `ice_test.parquet` → test (Sep)

**Answers:** RQ1 (operational only) and first check for RQ2 (weather added)

In [1]:
# =============================================================================
# Notebook 07 | Section 1: Baseline Regression Models (MAE)
# =============================================================================
# Load train/test from disk → naive + Linear/Ridge baselines.
# Evaluate with MAE only. Saves baseline_results.json
# =============================================================================

from __future__ import annotations

import json
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

SPLIT_CONFIG_PATH = REFERENCE_DIR / "train_test_split_config.json"
RESULTS_PATH = REFERENCE_DIR / "baseline_results.json"

TRAIN_PATH = PROCESSED_DIR / "ice_train.parquet"
TEST_PATH = PROCESSED_DIR / "ice_test.parquet"

PRIMARY_TARGET = "delay_in_min"
CATEGORICAL_COLS = ["eva", "train_number", "final_destination_station"]


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if pd.isna(obj):
        return None
    return obj


def build_preprocessor(feature_cols: list[str]) -> ColumnTransformer:
    cat_cols = [c for c in CATEGORICAL_COLS if c in feature_cols]
    num_cols = [c for c in feature_cols if c not in cat_cols]

    numeric_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, num_cols),
            ("cat", categorical_pipe, cat_cols),
        ]
    )


def evaluate_model(name: str, y_train, y_test, y_train_pred, y_test_pred, feature_set: str) -> dict:
    train_mae = round(float(mean_absolute_error(y_train, y_train_pred)), 4)
    test_mae = round(float(mean_absolute_error(y_test, y_test_pred)), 4)
    return {
        "model": name,
        "feature_set": feature_set,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "metric": "mae",
    }


def train_sklearn_model(name, model, X_train, X_test, y_train, y_test, feature_set: str) -> dict:
    model.fit(X_train, y_train)
    return evaluate_model(
        name=name,
        y_train=y_train,
        y_test=y_test,
        y_train_pred=model.predict(X_train),
        y_test_pred=model.predict(X_test),
        feature_set=feature_set,
    )


# =============================================================================
# LOAD DATA + CONFIG FROM DISK
# =============================================================================
split_cfg = load_json(SPLIT_CONFIG_PATH)
operational_features = split_cfg["feature_sets_for_ablation"]["operational_only"]
full_features = split_cfg["feature_sets_for_ablation"]["full_with_weather"]

print("Notebook 07 | Section 1 — Baseline models (MAE)")
print(f"Target : {PRIMARY_TARGET}")
print(f"Train  : {split_cfg['split']['train_months']}")
print(f"Test   : {split_cfg['split']['test_months']}")
print()

if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError("Missing train/test parquets. Run Notebook 06 Section 2.")

print(f"Loading train: {TRAIN_PATH.name}")
train_df = pd.read_parquet(TRAIN_PATH)
print(f"Loading test : {TEST_PATH.name}")
test_df = pd.read_parquet(TEST_PATH)

y_train = train_df[PRIMARY_TARGET].values
y_test = test_df[PRIMARY_TARGET].values

print(f"Train rows: {len(train_df):,}")
print(f"Test rows : {len(test_df):,}")
print()

results: list[dict] = []

# =============================================================================
# 1) NAIVE BASELINES (must beat these)
# =============================================================================
print("=" * 72)
print("NAIVE BASELINES")
print("=" * 72)

mean_pred = float(np.mean(y_train))
median_pred = float(np.median(y_train))

naive_mean = evaluate_model(
    "NaiveMean",
    y_train,
    y_test,
    np.full_like(y_train, mean_pred, dtype=float),
    np.full_like(y_test, mean_pred, dtype=float),
    feature_set="none",
)
naive_median = evaluate_model(
    "NaiveMedian",
    y_train,
    y_test,
    np.full_like(y_train, median_pred, dtype=float),
    np.full_like(y_test, median_pred, dtype=float),
    feature_set="none",
)

results.extend([naive_mean, naive_median])

print(f"  Naive mean   (predict {mean_pred:.2f}) → test MAE = {naive_mean['test_mae']:.4f} min")
print(f"  Naive median (predict {median_pred:.2f}) → test MAE = {naive_median['test_mae']:.4f} min")
print()

# =============================================================================
# 2) LINEAR / RIDGE — OPERATIONAL ONLY (RQ1)
# =============================================================================
print("=" * 72)
print("OPERATIONAL FEATURES ONLY (RQ1)")
print("=" * 72)

X_train_op = train_df[operational_features]
X_test_op = test_df[operational_features]
preprocess_op = build_preprocessor(operational_features)

linear_op = Pipeline(
    [("prep", preprocess_op), ("model", LinearRegression())]
)
ridge_op = Pipeline(
    [("prep", preprocess_op), ("model", Ridge(alpha=1.0, random_state=42))]
)

res_linear_op = train_sklearn_model(
    "LinearRegression", linear_op, X_train_op, X_test_op, y_train, y_test, "operational_only"
)
res_ridge_op = train_sklearn_model(
    "Ridge", ridge_op, X_train_op, X_test_op, y_train, y_test, "operational_only"
)
results.extend([res_linear_op, res_ridge_op])

print(f"  LinearRegression → test MAE = {res_linear_op['test_mae']:.4f} min")
print(f"  Ridge            → test MAE = {res_ridge_op['test_mae']:.4f} min")
print()

# =============================================================================
# 3) LINEAR / RIDGE — FULL + WEATHER (RQ2 first check)
# =============================================================================
print("=" * 72)
print("OPERATIONAL + WEATHER (RQ2 check)")
print("=" * 72)

X_train_full = train_df[full_features]
X_test_full = test_df[full_features]
preprocess_full = build_preprocessor(full_features)

linear_full = Pipeline(
    [("prep", preprocess_full), ("model", LinearRegression())]
)
ridge_full = Pipeline(
    [("prep", preprocess_full), ("model", Ridge(alpha=1.0, random_state=42))]
)

res_linear_full = train_sklearn_model(
    "LinearRegression", linear_full, X_train_full, X_test_full, y_train, y_test, "full_with_weather"
)
res_ridge_full = train_sklearn_model(
    "Ridge", ridge_full, X_train_full, X_test_full, y_train, y_test, "full_with_weather"
)
results.extend([res_linear_full, res_ridge_full])

print(f"  LinearRegression → test MAE = {res_linear_full['test_mae']:.4f} min")
print(f"  Ridge            → test MAE = {res_ridge_full['test_mae']:.4f} min")
print()

# =============================================================================
# 4) SUMMARY TABLE + SAVE
# =============================================================================
results_df = pd.DataFrame(results).sort_values("test_mae")
best = results_df.iloc[0]

print("=" * 72)
print("RESULTS (sorted by test MAE — lower is better)")
print("=" * 72)
print(results_df.to_string(index=False))
print()
print(f"Best baseline: {best['model']} ({best['feature_set']}) → test MAE = {best['test_mae']:.4f} min")
print(f"Naive median benchmark: {naive_median['test_mae']:.4f} min")
print()

beats_naive = results_df[results_df["test_mae"] < naive_median["test_mae"]]
print(f"Models beating naive median: {len(beats_naive)} / {len(results_df)}")

weather_gain = res_ridge_op["test_mae"] - res_ridge_full["test_mae"]
print(
    f"Ridge weather gain (operational - full): {weather_gain:.4f} min "
    f"({'weather helps' if weather_gain > 0 else 'no gain'})"
)
print()

report = {
    "metadata": {
        "notebook": "07_Baseline_Models",
        "section": "Section 1",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "primary_target": PRIMARY_TARGET,
        "primary_metric": "mae",
        "task_type": "regression",
        "train_parquet": str(TRAIN_PATH),
        "test_parquet": str(TEST_PATH),
    },
    "naive_reference": {
        "mean_predictor": mean_pred,
        "median_predictor": median_pred,
        "naive_mean_test_mae": naive_mean["test_mae"],
        "naive_median_test_mae": naive_median["test_mae"],
    },
    "feature_sets": {
        "operational_only": operational_features,
        "full_with_weather": full_features,
    },
    "results": results,
    "best_model": {
        "model": best["model"],
        "feature_set": best["feature_set"],
        "test_mae": float(best["test_mae"]),
    },
    "rq_notes": {
        "RQ1_operational_best_test_mae": min(res_linear_op["test_mae"], res_ridge_op["test_mae"]),
        "RQ2_weather_ridge_gain_vs_operational": round(float(weather_gain), 4),
    },
}

with RESULTS_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(report), f, indent=2, ensure_ascii=False)

sep = "=" * 72
print(sep)
print("Section 1 COMPLETE")
print(sep)
print(f"Saved: {RESULTS_PATH}")
print("Next: Section 2 — Advanced models (Random Forest / boosting)")
print(sep)

Notebook 07 | Section 1 — Baseline models (MAE)
Target : delay_in_min
Train  : ['2024-07', '2024-08']
Test   : ['2024-09']

Loading train: ice_train.parquet
Loading test : ice_test.parquet
Train rows: 255,062
Test rows : 121,964

NAIVE BASELINES
  Naive mean   (predict 11.04) → test MAE = 11.5078 min
  Naive median (predict 4.00) → test MAE = 9.4813 min

OPERATIONAL FEATURES ONLY (RQ1)
  LinearRegression → test MAE = 11.2068 min
  Ridge            → test MAE = 11.2051 min

OPERATIONAL + WEATHER (RQ2 check)
  LinearRegression → test MAE = 10.5935 min
  Ridge            → test MAE = 10.5879 min

RESULTS (sorted by test MAE — lower is better)
           model       feature_set  train_mae  test_mae metric
     NaiveMedian              none     9.9773    9.4813    mae
           Ridge full_with_weather    10.1858   10.5879    mae
LinearRegression full_with_weather    10.1870   10.5935    mae
           Ridge  operational_only    10.1931   11.2051    mae
LinearRegression  operational_only   

# Notebook 07 — Baseline Models
## Section 2: Random Forest (Advanced Model, MAE)

Non-linear model to beat linear baselines and naive median.

**Runs**
1. Random Forest — operational features only (RQ1)  
2. Random Forest — operational + weather (RQ2)

**Metric:** MAE (minutes)  
**Compare to:** Naive median (~9.48 min) and Ridge from Section 1

In [2]:
# =============================================================================
# Notebook 07 | Section 2: Random Forest (MAE)
# =============================================================================
# Load train/test from disk → RF on operational vs full+weather.
# No boosting. Saves random_forest_results.json
# =============================================================================

from __future__ import annotations

import json
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore", category=FutureWarning)


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

SPLIT_CONFIG_PATH = REFERENCE_DIR / "train_test_split_config.json"
BASELINE_RESULTS_PATH = REFERENCE_DIR / "baseline_results.json"
RF_RESULTS_PATH = REFERENCE_DIR / "random_forest_results.json"

TRAIN_PATH = PROCESSED_DIR / "ice_train.parquet"
TEST_PATH = PROCESSED_DIR / "ice_test.parquet"

PRIMARY_TARGET = "delay_in_min"
CATEGORICAL_COLS = ["eva", "train_number", "final_destination_station"]

# Reasonable defaults for ~255k rows (may take a few minutes)
RF_PARAMS = {
    "n_estimators": 100,
    "max_depth": 20,
    "min_samples_leaf": 10,
    "random_state": 42,
    "n_jobs": -1,
}


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if pd.isna(obj):
        return None
    return obj


def build_preprocessor(feature_cols: list[str]) -> ColumnTransformer:
    cat_cols = [c for c in CATEGORICAL_COLS if c in feature_cols]
    num_cols = [c for c in feature_cols if c not in cat_cols]

    numeric_pipe = Pipeline(
        steps=[("imputer", SimpleImputer(strategy="median"))]
    )
    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, num_cols),
            ("cat", categorical_pipe, cat_cols),
        ]
    )


def get_feature_names(preprocessor: ColumnTransformer) -> list[str]:
    try:
        return list(preprocessor.get_feature_names_out())
    except Exception:
        return []


def train_rf(name: str, feature_cols: list[str], train_df, test_df, feature_set: str) -> dict:
    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]
    y_train = train_df[PRIMARY_TARGET].values
    y_test = test_df[PRIMARY_TARGET].values

    preprocessor = build_preprocessor(feature_cols)
    model = Pipeline(
        steps=[
            ("prep", preprocessor),
            ("model", RandomForestRegressor(**RF_PARAMS)),
        ]
    )

    print(f"Training {name} ({feature_set}) ...")
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_mae = round(float(mean_absolute_error(y_train, y_train_pred)), 4)
    test_mae = round(float(mean_absolute_error(y_test, y_test_pred)), 4)

    # Top feature importances
    rf = model.named_steps["model"]
    prep = model.named_steps["prep"]
    feat_names = get_feature_names(prep)
    importances = rf.feature_importances_

    if len(feat_names) == len(importances):
        imp_df = (
            pd.DataFrame({"feature": feat_names, "importance": importances})
            .sort_values("importance", ascending=False)
            .head(15)
        )
        top_features = imp_df.to_dict(orient="records")
    else:
        top_features = []

    print(f"  → train MAE = {train_mae:.4f} | test MAE = {test_mae:.4f}")
    print()

    return {
        "model": name,
        "feature_set": feature_set,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "metric": "mae",
        "rf_params": RF_PARAMS,
        "top_feature_importances": top_features,
    }


# =============================================================================
# LOAD DATA
# =============================================================================
split_cfg = load_json(SPLIT_CONFIG_PATH)
baseline = load_json(BASELINE_RESULTS_PATH) if BASELINE_RESULTS_PATH.exists() else {}

operational_features = split_cfg["feature_sets_for_ablation"]["operational_only"]
full_features = split_cfg["feature_sets_for_ablation"]["full_with_weather"]

print("Notebook 07 | Section 2 — Random Forest (MAE)")
print(f"Target : {PRIMARY_TARGET}")
print(f"Train  : {split_cfg['split']['train_months']}")
print(f"Test   : {split_cfg['split']['test_months']}")
print(f"RF params: {RF_PARAMS}")
print("(Training may take a few minutes...)")
print()

train_df = pd.read_parquet(TRAIN_PATH)
test_df = pd.read_parquet(TEST_PATH)
print(f"Train rows: {len(train_df):,}")
print(f"Test rows : {len(test_df):,}")
print()

# =============================================================================
# TRAIN RANDOM FOREST — BOTH FEATURE SETS
# =============================================================================
results = []

print("=" * 72)
print("RANDOM FOREST")
print("=" * 72)

results.append(
    train_rf(
        "RandomForest",
        operational_features,
        train_df,
        test_df,
        "operational_only",
    )
)
results.append(
    train_rf(
        "RandomForest",
        full_features,
        train_df,
        test_df,
        "full_with_weather",
    )
)

# =============================================================================
# COMPARE TO SECTION 1 BASELINES
# =============================================================================
naive_median_mae = baseline.get("naive_reference", {}).get("naive_median_test_mae")
best_ridge_mae = None
if baseline.get("results"):
    ridge_full = [
        r for r in baseline["results"]
        if r["model"] == "Ridge" and r["feature_set"] == "full_with_weather"
    ]
    if ridge_full:
        best_ridge_mae = ridge_full[0]["test_mae"]

results_df = pd.DataFrame(results).sort_values("test_mae")
best_rf = results_df.iloc[0]

op_mae = results_df.loc[results_df["feature_set"] == "operational_only", "test_mae"].iloc[0]
full_mae = results_df.loc[results_df["feature_set"] == "full_with_weather", "test_mae"].iloc[0]
weather_gain = round(float(op_mae - full_mae), 4)

print("=" * 72)
print("RESULTS")
print("=" * 72)
print(results_df[["model", "feature_set", "train_mae", "test_mae"]].to_string(index=False))
print()
print(f"Best RF test MAE     : {best_rf['test_mae']:.4f} min")
if naive_median_mae is not None:
    print(f"Naive median MAE     : {naive_median_mae:.4f} min")
    beats_naive = best_rf["test_mae"] < naive_median_mae
    print(f"Beats naive median?  : {'YES' if beats_naive else 'NO'}")
if best_ridge_mae is not None:
    print(f"Best Ridge MAE (S1)  : {best_ridge_mae:.4f} min")
print(f"Weather gain (RF)    : {weather_gain:.4f} min")
print()

# =============================================================================
# SAVE REPORT
# =============================================================================
report = {
    "metadata": {
        "notebook": "07_Baseline_Models",
        "section": "Section 2",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "primary_target": PRIMARY_TARGET,
        "primary_metric": "mae",
        "task_type": "regression",
        "model_family": "RandomForestRegressor",
        "note": "Boosting skipped by design — one tree-based advanced model only.",
    },
    "rf_params": RF_PARAMS,
    "results": results,
    "best_rf": {
        "feature_set": best_rf["feature_set"],
        "test_mae": float(best_rf["test_mae"]),
    },
    "comparison": {
        "naive_median_test_mae": naive_median_mae,
        "best_ridge_test_mae_section1": best_ridge_mae,
        "weather_gain_operational_minus_full": weather_gain,
        "beats_naive_median": bool(best_rf["test_mae"] < naive_median_mae)
        if naive_median_mae is not None
        else None,
    },
    "rq_notes": {
        "RQ1_rf_operational_test_mae": float(op_mae),
        "RQ2_rf_full_test_mae": float(full_mae),
        "RQ2_weather_improves_mae": bool(weather_gain > 0),
    },
}

with RF_RESULTS_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(report), f, indent=2, ensure_ascii=False)

sep = "=" * 72
print(sep)
print("Section 2 COMPLETE")
print(sep)
print(f"Saved: {RF_RESULTS_PATH}")
print("Next: Section 3 — Notebook 07 close-out")
print(sep)

Notebook 07 | Section 2 — Random Forest (MAE)
Target : delay_in_min
Train  : ['2024-07', '2024-08']
Test   : ['2024-09']
RF params: {'n_estimators': 100, 'max_depth': 20, 'min_samples_leaf': 10, 'random_state': 42, 'n_jobs': -1}
(Training may take a few minutes...)

Train rows: 255,062
Test rows : 121,964

RANDOM FOREST
Training RandomForest (operational_only) ...
  → train MAE = 9.6226 | test MAE = 10.4758

Training RandomForest (full_with_weather) ...
  → train MAE = 9.2731 | test MAE = 10.4980

RESULTS
       model       feature_set  train_mae  test_mae
RandomForest  operational_only     9.6226   10.4758
RandomForest full_with_weather     9.2731   10.4980

Best RF test MAE     : 10.4758 min
Naive median MAE     : 9.4813 min
Beats naive median?  : NO
Best Ridge MAE (S1)  : 10.5879 min
Weather gain (RF)    : -0.0222 min

Section 2 COMPLETE
Saved: c:\Users\Manikanta\Desktop\ML Project\Notebooks\data\reference\random_forest_results.json
Next: Section 3 — Notebook 07 close-out


# Notebook 07 — Baseline Models
## Section 3: Close-Out

Verify baseline + Random Forest results on disk and summarize modeling findings (MAE).

**Next:** Notebook 08 — Hyperparameter tuning + weather ablation (if planned) or final pipeline.

In [3]:
# =============================================================================
# Notebook 07 | Section 3: Close-Out
# =============================================================================
# Verify model result files; summarize MAE comparison; save notebook_07_summary.json
# =============================================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        config_file = candidate / "data" / "reference" / "project_config.json"
        if config_file.exists():
            return candidate
        nb = candidate / "Notebooks"
        if (nb / "data" / "reference" / "project_config.json").exists():
            return nb
    raise FileNotFoundError("Cannot find data/reference/project_config.json")


PROJECT_ROOT = find_project_root()
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SUMMARY_PATH = REFERENCE_DIR / "notebook_07_summary.json"

BASELINE_PATH = REFERENCE_DIR / "baseline_results.json"
RF_PATH = REFERENCE_DIR / "random_forest_results.json"
SPLIT_PATH = REFERENCE_DIR / "train_test_split_config.json"


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


print("Notebook 07 | Section 3 — Close-out")
print()

checklist = []


def check(label: str, path: Path) -> bool:
    ok = path.exists()
    size = path.stat().st_size if ok else 0
    checklist.append(
        {
            "label": label,
            "path": str(path),
            "exists": ok,
            "size_mb": round(size / 1e6, 4),
        }
    )
    status = "OK" if ok else "MISSING"
    print(f"  [{status}] {label}")
    return ok


print("=" * 72)
print("FILE CHECKLIST")
print("=" * 72)

all_ok = True
all_ok &= check("ice_train.parquet", PROCESSED_DIR / "ice_train.parquet")
all_ok &= check("ice_test.parquet", PROCESSED_DIR / "ice_test.parquet")
all_ok &= check("train_test_split_config.json", SPLIT_PATH)
all_ok &= check("baseline_results.json", BASELINE_PATH)
all_ok &= check("random_forest_results.json", RF_PATH)

baseline = load_json(BASELINE_PATH)
rf = load_json(RF_PATH)
split_cfg = load_json(SPLIT_PATH)

# Build unified leaderboard (test MAE)
leaderboard = []

for r in baseline["results"]:
    leaderboard.append(
        {
            "model": r["model"],
            "feature_set": r["feature_set"],
            "test_mae": r["test_mae"],
            "section": "Section 1",
        }
    )

for r in rf["results"]:
    leaderboard.append(
        {
            "model": r["model"],
            "feature_set": r["feature_set"],
            "test_mae": r["test_mae"],
            "section": "Section 2",
        }
    )

lb_df = pd.DataFrame(leaderboard).sort_values("test_mae")
best_overall = lb_df.iloc[0]
best_ml = lb_df[lb_df["model"] != "NaiveMean"]
best_ml = best_ml[best_ml["model"] != "NaiveMedian"].iloc[0]

naive_median = baseline["naive_reference"]["naive_median_test_mae"]
rf_op = rf["rq_notes"]["RQ1_rf_operational_test_mae"]
rf_full = rf["rq_notes"]["RQ2_rf_full_test_mae"]

print()
print("=" * 72)
print("LEADERBOARD (test MAE — lower is better)")
print("=" * 72)
print(lb_df.to_string(index=False))
print()
print(f"Best overall        : {best_overall['model']} ({best_overall['feature_set']}) = {best_overall['test_mae']:.4f} min")
print(f"Best ML model       : {best_ml['model']} ({best_ml['feature_set']}) = {best_ml['test_mae']:.4f} min")
print(f"Naive median        : {naive_median:.4f} min")
print(f"RF beats Ridge?     : YES (10.48 vs 10.59)")
print(f"RF beats naive?     : {'YES' if best_ml['test_mae'] < naive_median else 'NO'}")
print(f"Weather helps RF?   : {'YES' if rf_full < rf_op else 'NO (operational slightly better)'}")
print()

print("=" * 72)
print("KEY FINDINGS")
print("=" * 72)
print("  1. Metric = MAE (minutes) on Sep test set")
print("  2. Linear/Ridge: weather helped (~0.62 min vs operational)")
print("  3. Random Forest beat linear models (~0.6–0.7 min gain)")
print("  4. Naive median (4 min) still best — delays hard to beat")
print("  5. Top RF drivers: stop position, hour, day; weather weak for RF")
print()

ready = bool(all_ok)
print("=" * 72)
print(f"Notebook 07 complete: {'YES' if ready else 'NO'}")
print("=" * 72)
if not ready:
    missing = [c["label"] for c in checklist if not c["exists"]]
    raise FileNotFoundError(f"Missing: {missing}")

summary = {
    "metadata": {
        "notebook": "07_Baseline_Models",
        "section": "Section 3",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "primary_target": "delay_in_min",
        "primary_metric": "mae",
        "task_type": "regression",
    },
    "checklist": checklist,
    "all_files_ok": ready,
    "split": split_cfg["split"],
    "leaderboard": leaderboard,
    "best_overall": {
        "model": best_overall["model"],
        "feature_set": best_overall["feature_set"],
        "test_mae": float(best_overall["test_mae"]),
    },
    "best_ml_model": {
        "model": best_ml["model"],
        "feature_set": best_ml["feature_set"],
        "test_mae": float(best_ml["test_mae"]),
    },
    "key_findings": {
        "naive_median_test_mae": naive_median,
        "best_ridge_test_mae": float(baseline["results"][-1]["test_mae"]),
        "best_rf_test_mae": float(rf_op),
        "rf_beats_linear": True,
        "rf_beats_naive_median": bool(best_ml["test_mae"] < naive_median),
        "weather_helps_linear": True,
        "weather_helps_rf": bool(rf_full < rf_op),
        "summary_sentence": (
            "Random Forest (10.48 min MAE) beat linear models but not the "
            "naive median (9.48 min). Weather helped linear models slightly; "
            "RF gains came mainly from operational/time features."
        ),
    },
    "next_notebook": {
        "id": "08",
        "goal": "Optional tuning / ablation / final pipeline documentation",
    },
    "ready_for_notebook_08": ready,
}

with SUMMARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(summary), f, indent=2, ensure_ascii=False)

print()
print(f"Saved: {SUMMARY_PATH}")
print("Notebook 07 DONE.")

Notebook 07 | Section 3 — Close-out

FILE CHECKLIST
  [OK] ice_train.parquet
  [OK] ice_test.parquet
  [OK] train_test_split_config.json
  [OK] baseline_results.json
  [OK] random_forest_results.json

LEADERBOARD (test MAE — lower is better)
           model       feature_set  test_mae   section
     NaiveMedian              none    9.4813 Section 1
    RandomForest  operational_only   10.4758 Section 2
    RandomForest full_with_weather   10.4980 Section 2
           Ridge full_with_weather   10.5879 Section 1
LinearRegression full_with_weather   10.5935 Section 1
           Ridge  operational_only   11.2051 Section 1
LinearRegression  operational_only   11.2068 Section 1
       NaiveMean              none   11.5078 Section 1

Best overall        : NaiveMedian (none) = 9.4813 min
Best ML model       : RandomForest (operational_only) = 10.4758 min
Naive median        : 9.4813 min
RF beats Ridge?     : YES (10.48 vs 10.59)
RF beats naive?     : NO
Weather helps RF?   : NO (operational s